# Fuzzy Kart: racing SuperTuxKart with fuzzy logic

In notebook 02 a fuzzy driver raced the lecture circuits. Here it races
[SuperTuxKart](https://supertuxkart.net), the open-source kart game: real 3-D tracks,
grass, walls, and AI opponents. By the end you will:

1. read a working fuzzy driver and watch it race,
2. look inside one of its decisions, rule by rule,
3. change its rules and measure the effect,
4. tune its numbers automatically,
5. drive it in the **class race**, live from this notebook.

Everything runs on **Google Colab** (*Runtime → Run all*) or locally.

## 0. Setup

On Colab this installs `fuzzy_torch` with SuperTuxKart and fetches the example files.
The first race downloads the game itself (about 700 MB, a minute or two on Colab);
later races start in seconds.

In [ ]:
import os, sys, pathlib, importlib

REF = 'supertuxkart'          # the branch to install; 'main' once it is merged
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install "torch-fis[kart] @ git+https://github.com/mdrs-thiago/fuzzy_torch.git@{REF}"
    !git clone -q --depth 1 --branch {REF} https://github.com/mdrs-thiago/fuzzy_torch.git /content/fuzzy_torch_repo
    REPO = pathlib.Path('/content/fuzzy_torch_repo')
else:
    # This notebook lives in fuzzy_torch/examples/notebooks/.
    REPO = pathlib.Path.cwd().resolve().parents[1]
    try:
        importlib.import_module('fuzzy_torch')
    except ModuleNotFoundError:
        sys.path.insert(0, str(REPO.parent))   # flat package: put its parent on sys.path

import collections, copy, json, subprocess, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import fuzzy_torch as ft
from fuzzy_torch.envs import SuperTuxKartEnv
from fuzzy_torch.envs.controllers import ConfigRaceDriver
from fuzzy_torch.envs.controllers.racing import race_inputs, system_inputs

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
WORK = pathlib.Path('kart_work')              # your driver files and videos go here
WORK.mkdir(exist_ok=True)
print('fuzzy_torch', ft.__version__, '| repo at', REPO, '| files in', WORK.resolve())

## 1. The starter driver

A driver is two fuzzy systems in a JSON file: **steering** and **throttle**. Every
0.05 s they read six numbers about the kart and the road and answer
`[throttle, steering]`, both in `[-1, 1]`:

| input | meaning | typical values |
|---|---|---|
| `cross_track_error` | distance from the middle of the road, in half road widths (±1 = road edge), positive = left of it | −0.5 … 0.5 |
| `heading_error` | road direction minus kart heading (rad), positive = road turns away to the left | −0.2 … 0.2 |
| `speed` | forward speed, top speed reads 4 | 2 … 3.5 |
| `yaw_rate` | how fast the kart turns, positive = left | −1.3 … 1.3 |
| `lookahead_curvature` | the bend about 10 m ahead, positive = left bend | −0.4 … 0.4 |
| `curve` | how sharp that bend is: `abs(lookahead_curvature)` | 0 … 0.45 |

The starter steers with **Mamdani** rules on three inputs, and sets the throttle with
**zero-order TSK** rules (each rule outputs a number, so full throttle can be exactly 1).

In [ ]:
starter = json.loads((REPO / 'examples/race_control/kart_starter.json').read_text())
driver = ConfigRaceDriver(starter)
for name, system in (('steering', driver.steering), ('throttle', driver.throttle)):
    print(f"{name:9s} {system.mode:8s} inputs {system.input_order}  rules {len(system.rules)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, name in zip(axes, driver.steering.input_order):
    driver.steering.input_variables[name].plot(ax=ax)
fig.suptitle('steering: the inputs and their terms'); fig.tight_layout()

## 2. Race it

`run_race` drives one race and keeps a log of every step. Without AI karts a race
repeats exactly, which makes it a fair test; with them, collisions make each race a
little different.

In [ ]:
def run_race(driver, track='zengarden', opponents=0, lag=0, seed=0):
    """One race; `lag` delays every action by that many 0.05 s steps, as over a network."""
    env = SuperTuxKartEnv(track, opponents=opponents)
    obs, _ = env.reset(seed=seed)
    delayed = collections.deque([np.zeros(2, np.float32)] * lag)
    log = collections.defaultdict(list)
    while True:
        action = driver(obs)
        delayed.append(action)
        obs, _, terminated, truncated, info = env.step(delayed.popleft())
        for key, value in (('obs', obs), ('action', action), ('xy', info['xy']),
                           ('off_track', info['off_track'])):
            log[key].append(value)
        if terminated or truncated:
            break
    centre = np.vstack([env.track.start, env.track.start[:1]])
    env.close()
    return {'track': track, 'time': info['race_time'], 'position': info['position'],
            'karts': info['num_karts'], 'rescues': info['rescues'], 'centre': centre,
            'off_track': float(np.mean(log['off_track'])),
            **{k: np.asarray(v) for k, v in log.items() if k != 'off_track'}}

def summary(race):
    result = f"{race['time']:.2f} s" if race['time'] else 'did not finish'
    return (f"{race['track']}: {result}, position {race['position']}/{race['karts']}, "
            f"off the road {100 * race['off_track']:.0f}% of the time, rescues {race['rescues']}")

race = run_race(driver, 'zengarden')
print(summary(race))

In [ ]:
def plot_race(race, title=''):
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [1, 1.4]})
    ax.plot(*race['centre'].T, color='#bbb', lw=10, solid_capstyle='round')
    points = ax.scatter(*race['xy'].T, c=race['obs'][:, 2], s=3, cmap='viridis')
    fig.colorbar(points, ax=ax, label='speed (4 = top)')
    ax.set_aspect('equal'); ax.set_title(title or summary(race), fontsize=9)
    t = np.arange(len(race['obs'])) * 0.05
    bx.plot(t, race['obs'][:, 0], label='cross_track_error', lw=1)
    bx.plot(t, race['obs'][:, 4] * 5, label='lookahead_curvature x5', lw=1)
    bx.plot(t, race['action'][:, 1], label='steering', lw=1)
    bx.plot(t, race['action'][:, 0], label='throttle', lw=1)
    bx.axhline(1, ls=':', c='k'); bx.axhline(-1, ls=':', c='k')
    bx.set_xlabel('time (s)'); bx.legend(fontsize=8, ncol=2); fig.tight_layout()

plot_race(race)

### Look inside one decision

Pick the moment the driver steered hardest and replay the steering system's
inference there: which terms each input activated, which rules fired, and how the
output was assembled.

In [ ]:
k = int(np.argmax(np.abs(race['action'][:, 1])))
values = race_inputs(race['obs'][k])
print(f"t = {k * 0.05:.2f} s:", {n: round(values[n], 3) for n in driver.steering.input_order})
inputs = system_inputs(driver.steering, values)
driver.steering.explain(inputs)
driver.steering.plot_inference(inputs);

### Watch it (optional)

The game renders in a separate process, so a machine without graphics (as Colab
often is) shows a message here instead of stopping the notebook.

In [ ]:
from IPython.display import HTML
import base64

def video(spec, track='zengarden', opponents=3, name='race.webp'):
    path, out = WORK / 'driver_for_video.json', WORK / name
    path.write_text(json.dumps(spec))
    command = [sys.executable, str(REPO / 'examples/supertuxkart_demo.py'), str(path),
               '--track', track, '--opponents', str(opponents), '--gif', str(out), '--every', '4']
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0 or not out.exists():
        print('No video on this machine (the game could not start its graphics):')
        print('\n'.join(done.stderr.strip().splitlines()[-3:]))
        return None
    data = base64.b64encode(out.read_bytes()).decode()
    return HTML(f'<img src="data:image/webp;base64,{data}" width="640">')

video(starter)

## 3. Your driver

Start from a copy of the starter and change it. A few ideas:

- Its heading terms are very narrow (`centre` is ±0.1 rad). What happens with ±0.3?
- The throttle ignores `yaw_rate`. Could braking while the kart is spinning help?
- Add `lookahead_curvature` to the throttle so it brakes *before* a sharp bend.

Then race it on several tracks: a driver that only works on one track is overfitted.
The example change below is a real experiment: in our test it made the driver
slightly *slower*. Can you find a change that makes it faster?

In [ ]:
my_spec = copy.deepcopy(starter)
my_spec['description'] = 'my driver'

# Example change: wider heading terms. Edit freely, then re-run the cells below.
heading = my_spec['steering']['inputs']['heading_error']['terms']
heading['right'] = ['trap', -1.6, -1.6, -0.3, 0]
heading['centre'] = ['tri', -0.3, 0, 0.3]
heading['left'] = ['trap', 0, 0.3, 1.6, 1.6]

my_driver = ConfigRaceDriver(my_spec)          # checks the spec; errors say where the problem is
_ = (WORK / 'my_driver.json').write_text(json.dumps(my_spec, indent=1))

In [ ]:
TRACKS = ['lighthouse', 'zengarden', 'hacienda']

def compare(specs, tracks=TRACKS):
    rows = {name: [run_race(ConfigRaceDriver(spec), t) for t in tracks] for name, spec in specs.items()}
    print(f"{'':12s}" + ''.join(f"{t:>14s}" for t in tracks) + f"{'total':>10s}")
    for name, races in rows.items():
        times = [r['time'] or float('nan') for r in races]
        print(f"{name:12s}" + ''.join(f"{v:13.2f}s" for v in times) + f"{sum(times):9.2f}s")
    return rows

results = compare({'starter': starter, 'my driver': my_spec})

## 4. Tune it automatically

`tune` changes only the **numbers** (term breakpoints and TSK outputs), never your
rules, so you can still explain every rule. It keeps a change only when the total
time on the tuning tracks drops. Each try races every tuning track once, so the
budget below takes several minutes on Colab.

If you will race from this notebook, tune with `LAG` set to the suggestion that
section 5 prints for your connection.

In [ ]:
sys.path.insert(0, str(REPO / 'examples'))
from kart_tune import tune, race_times

BUDGET, LAG = 15, 0
tuned_spec, tuned_times, start_times = tune(my_spec, TRACKS, budget=BUDGET, lag=LAG)
_ = (WORK / 'my_driver_tuned.json').write_text(json.dumps(tuned_spec, indent=1))
print(f"{sum(start_times):.1f} s -> {sum(tuned_times):.1f} s on {', '.join(TRACKS)}")

In [ ]:
CHECK = ['cornfield_crossing', 'snowmountain', 'scotland']     # tracks it was not tuned on
before, after = race_times(my_spec, CHECK, LAG), race_times(tuned_spec, CHECK, LAG)
for track, b, a in zip(CHECK, before, after):
    print(f"{track:20s} {b:6.1f} s -> {a:6.1f} s")
if tuned_spec == my_spec:
    print('unchanged: tuning found nothing faster (try a larger BUDGET)')
else:
    print('better on new tracks too' if sum(after) < sum(before) else 'it fitted the tuning tracks only')

## 5. The class race

Your instructor runs the race and shares the screen. The top of the race screen shows
the **address** and the **join code**. Fill them in below, with your name, and run the
two cells. Your driver drives from this notebook: keep the tab open until the race ends.

To submit a JSON file instead (for the Race Control leaderboard), download
`kart_work/my_driver_tuned.json` from the file panel and upload it on the page your
instructor gives.

In [ ]:
SERVER = ''      # e.g. 'https://abcd-1234.ngrok-free.app', as shown on the race screen
CODE = ''        # the join code shown on the race screen
NAME = ''        # your name on the standings
RACE_WITH = tuned_spec

from fuzzy_torch.envs.kart_net import join
connection = None
if not (SERVER and CODE and NAME):
    print('Fill in SERVER, CODE and NAME above, then run this cell again.')
else:
    connection = join(SERVER, NAME, CODE, ConfigRaceDriver(RACE_WITH), RACE_WITH)
    ms = connection.round_trip()
    print(f"Joined as {connection.name}. Round trip to the race: {ms:.0f} ms.")
    if ms > 200:
        print('That is slow for racing: your kart reacts late. '
              f'Tune with LAG = {connection.suggested_lag(ms)} for a driver that copes better.')

In [ ]:
from IPython.display import display

def race_live(connection):
    status = display('Waiting for the race to start...', display_id=True)
    shown = [0.0]

    def on_message(message, action):
        kind = message['type']
        if kind == 'obs' and time.monotonic() - shown[0] > 0.5:
            shown[0] = time.monotonic()
            status.update(f"P{message.get('position')}/{message.get('karts')}  lap {message.get('lap')}  "
                          f"{message.get('time', 0):5.1f} s  speed {message['obs'][2]:.2f}  "
                          f"throttle {action[0]:+.2f}  steering {action[1]:+.2f}")
        elif kind == 'start':
            status.update(f"Racing: {message['race']} on {message['track']}")
        elif kind == 'wait':
            status.update(f"Watching {message['race']}: you race in a later heat")
        elif kind == 'end':
            print(f"{message['race']} results:")
            for row in message['standings']:
                result = f"{row['race_time']:.2f} s" if row['race_time'] else f"{100 * row['progress']:.0f}% done"
                print(f"  {row['position']:2d}. {row['name']:24s} {result}")

    try:
        connection.run(on_message)
    finally:
        connection.close()
    print('The race server closed the connection.')

if connection is not None:
    race_live(connection)